# 01 · Workflow — pull the latest state and run nodes

This notebook is the day-to-day loop, and the main way to run nodes in IQCC Learn:

1. download the **latest QUAM state and wiring** from IQCC,
2. run a chain of nodes — resonator spectroscopy → qubit spectroscopy → power Rabi,
3. look at each node's data and fit.

If you have not yet seen what a node is made of, start with `00_start_here.ipynb`.

> 🧩 Packages: **QUAlibrate** ([docs](https://qua-platform.github.io/qualibrate/)) runs the nodes; **QUAM** ([docs](https://qua-platform.github.io/quam/)) holds the state they read and update;
> 💎 **QUA** ([docs](https://docs.quantum-machines.co/latest/)) is the program each node runs on the OPX.

## 1 · Settings

In [ ]:
QUBITS = ["q1"]     # qubits to calibrate
SIMULATE = False    # True: run on the OPX simulator instead of the hardware

## 2 · Backend and state location
Read from `pyproject.toml`, so it is the same in every checkout.

In [ ]:
import tomllib
from pathlib import Path

# The notebook lives in the checkout root, so the kernel's working directory is the checkout.
with open("pyproject.toml", "rb") as f:
    BACKEND = tomllib.load(f)["tool"]["iqcc"]["project-name"]
STATE_PATH = Path("config/quam-state") / BACKEND
print(f"backend: {BACKEND}\nstate:   {STATE_PATH.resolve()}")

## 3 · Download the latest state and wiring
Uses `iqcc_cloud_client` (IQCC credentials must already be configured). **This overwrites the local `state.json` and `wiring.json`.**

In [ ]:
import json
from iqcc_cloud_client import IQCC_Cloud

qc = IQCC_Cloud(BACKEND)
latest_wiring = qc.state.get_latest("wiring")
latest_state = qc.state.get_latest("state")

STATE_PATH.mkdir(parents=True, exist_ok=True)
with open(STATE_PATH / "wiring.json", "w") as f:
    json.dump(latest_wiring.data, f, indent=4)
with open(STATE_PATH / "state.json", "w") as f:
    json.dump(latest_state.data, f, indent=4)
print("Wrote wiring.json and state.json to", STATE_PATH)

## 4 · Open the calibration library
> 🧩 **QUAlibrate** — [docs](https://qua-platform.github.io/qualibrate/) · `QualibrationLibrary` scans `calibrations/` and registers every node by name.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display
from qualibrate import QualibrationLibrary

library = QualibrationLibrary.get_active_library()
print("available nodes:")
for name in sorted(library.nodes):
    print("  ", name)


def show(node):
    """Print outcomes and fit results, then show the figures a node saved."""
    print("outcomes:", node.outcomes)
    for qubit, result in node.results.get("fit_results", {}).items():
        print(qubit, {k: v for k, v in result.items() if not hasattr(v, "__len__") or isinstance(v, str)})
    for name, fig in node.results.get("figures", {}).items():
        print("figure:", name)
        display(fig)
    plt.close("all")

## 5 · Resonator spectroscopy
Finds the resonator frequency and updates `resonator.f_01` / `RF_frequency`.

> 💎 **QUA** — `02a_resonator_spectroscopy` builds and runs a QUA program on the OPX (see `00_start_here.ipynb` §6 for how one is written).
> 🧩 **QUAlibrate** — `library.nodes[...].run(...)` executes all the node's run-actions, overriding the parameters you pass.
> Each node updates the state, so the next one starts from the improved values.

In [ ]:
node = library.nodes["02a_resonator_spectroscopy"].run(qubits=QUBITS, simulate=SIMULATE)
show(node)

## 6 · Qubit spectroscopy
Sweeps the drive frequency to find the qubit transition. Needs the resonator frequency from the step before.

> 💎 **QUA** — `03a_qubit_spectroscopy` builds and runs a QUA program on the OPX (see `00_start_here.ipynb` §6 for how one is written).
> 🧩 **QUAlibrate** — `library.nodes[...].run(...)` executes all the node's run-actions, overriding the parameters you pass.
> Each node updates the state, so the next one starts from the improved values.

In [ ]:
node = library.nodes["03a_qubit_spectroscopy"].run(qubits=QUBITS, simulate=SIMULATE)
show(node)

## 7 · Power Rabi
Sweeps the pulse amplitude to calibrate the π-pulse. Needs the qubit frequency from the step before.

> 💎 **QUA** — `04b_power_rabi` builds and runs a QUA program on the OPX (see `00_start_here.ipynb` §6 for how one is written).
> 🧩 **QUAlibrate** — `library.nodes[...].run(...)` executes all the node's run-actions, overriding the parameters you pass.
> Each node updates the state, so the next one starts from the improved values.

In [ ]:
node = library.nodes["04b_power_rabi"].run(qubits=QUBITS, simulate=SIMULATE)
show(node)

## 8 · Optional: the same chain as a graph
> 🧩 **QUAlibrate** — a `QualibrationGraph` runs nodes in order and handles failed nodes for you
> (see `calibrations/tunable/` for real examples, if your checkout has them). Uncomment to try.

In [ ]:
# from qualibrate import QualibrationGraph
# from qualibrate.core.orchestration.basic_orchestrator import BasicOrchestrator
# from qualibrate.core.parameters import GraphParameters
#
# class GraphParams(GraphParameters):
#     qubits: list[str] = QUBITS
#
# g = QualibrationGraph(
#     name="notebook_chain",
#     parameters=GraphParams(),
#     nodes={n: library.nodes[n].copy(name=n) for n in ["02a_resonator_spectroscopy", "03a_qubit_spectroscopy", "04b_power_rabi"]},
#     connectivity=[("02a_resonator_spectroscopy", "03a_qubit_spectroscopy"), ("03a_qubit_spectroscopy", "04b_power_rabi")],
#     orchestrator=BasicOrchestrator(skip_failed=False),
# )
# g.run()